# 05 - DataForSEO: Search Volumes & Commercial Intent

This notebook extracts absolute search volumes and commercial intent metrics (CPC, Competition) using the DataForSEO API.

**Strategy for Machine Learning**:  
Unlike Google Trends (which provides a relative index of general interest), this dataset provides **absolute magnitudes** and **commercial variables** like `cpc` and `competition`.

## 1. Imports

In [7]:
import time
import requests
import pandas as pd

from pathlib import Path
from requests.auth import HTTPBasicAuth

from dotenv import load_dotenv
import os

## 2. Configuration

### 2.1 Load environment variables

In [8]:
load_dotenv()

DATAFORSEO_LOGIN = os.getenv("DATAFORSEO_LOGIN")
DATAFORSEO_PASSWORD = os.getenv("DATAFORSEO_PASSWORD")

### 2.2 Paths

In [ ]:
INPUT_PATH = Path("../outputs/keywords")
INTERIM_PATH = Path("../data/interim")
OUTPUT_PATH = Path("../outputs/google_ads")

INTERIM_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = (
    INTERIM_PATH /
    "05_dataforseo_monthly_checkpoint_v2.parquet"
)

FINAL_PATH = (
    INTERIM_PATH /
    "05_dataforseo_monthly_v2.parquet"
)

### 2.3 API (LABS ENDPOINT)

In [10]:
BATCH_SIZE = 700
LOCATION_CODE = 2724   # Spain
LANGUAGE_CODE = "es"

## 3. Target Data Ingestion & Checkpoint Management
Loads the target keywords and checks if a previous extraction was interrupted to avoid paying for duplicated API calls.

In [11]:
# Load the keywords

df_keywords = pd.read_parquet(
    INPUT_PATH / "04_keywords_with_prefix.parquet"
    )

target_keywords = df_keywords["google_ads_keyword"].dropna().unique().tolist()

print(f"Total target keywords: {len(target_keywords)}")


# 2. Checkpoint management

if CHECKPOINT_FILE.exists():
    df_checkpoint = pd.read_parquet(CHECKPOINT_FILE)
    downloaded_keywords = df_checkpoint["search_term"].unique().tolist()
    missing_queries = [kw for kw in target_keywords if kw not in downloaded_keywords]
    print(f"✔ Checkpoint found. Resuming extraction. {len(missing_queries)} keywords left.")
else:
    df_checkpoint = pd.DataFrame()
    missing_queries = target_keywords
    print(f"No checkpoint found. Starting new extraction for {len(missing_queries)} keywords.")

Total target keywords: 220
No checkpoint found. Starting new extraction for 220 keywords.


## 4. Extraction Engine Funtion

In [12]:
def fetch_dataforseo_labs_volumes(keywords_batch: list, location: int, language: str) -> list:
    """
    Fetches historical search volume data from DataForSEO Labs API.

    Args:
        keywords_batch (list): Keywords to query.
        location (int): Geographical location code (e.g., 2724 for Spain).
        language (str): Language code (e.g., 'es' for Spanish).

    Returns:
        list: Flattened dictionary containing period, volume, cpc, and competition.
    """
    post_data = [{
        "keywords": keywords_batch,
        "location_code": location,
        "language_code": language,
        "include_serp_info": False
    }]
    
    try:
        response = requests.post(
            "https://api.dataforseo.com/v3/dataforseo_labs/google/historical_search_volume/live",
            auth=HTTPBasicAuth(DATAFORSEO_LOGIN, DATAFORSEO_PASSWORD),
            json=post_data
        )
        response.raise_for_status()
        result_data = response.json()
        
        all_rows = []
        
        # Parse the JSON response based on Labs structure
        if "tasks" in result_data and len(result_data["tasks"]) > 0:
            task_result = result_data["tasks"][0].get("result", [])
            
            # The Labs structure typically nests the data inside 'items'
            if task_result and "items" in task_result[0]:
                for item in task_result[0]["items"]:
                    keyword = item.get("keyword")
                    
                    # Extract commercial metrics from keyword_info
                    keyword_info = item.get("keyword_info", {})
                    cpc = keyword_info.get("cpc")
                    competition = keyword_info.get("competition")
                    comp_level = keyword_info.get("competition_level")
                    
                    monthly_data = keyword_info.get("monthly_searches", [])
                    
                    for m_data in monthly_data:
                        year = m_data.get("year")
                        month = m_data.get("month")
                        sv = m_data.get("search_volume")
                        
                        if year and month:
                            date_str = f"{year}-{month:02d}-01"
                            all_rows.append({
                                "period": date_str,
                                "search_term": keyword,
                                "monthly_searches": sv,
                                "competition": competition,
                                "competition_level": comp_level,
                                "cpc": cpc
                            })
        return all_rows

    except requests.exceptions.RequestException as e:
        print(f"API Request failed: {e}")
        return []


## 5. Execution Pipeline

In [ ]:
RUN_API = False  # ⚠ Set to True to execute — costs real API credits

if RUN_API:
    if missing_queries:
        total_batches = (len(missing_queries) // BATCH_SIZE) + 1
        
        for i in range(0, len(missing_queries), BATCH_SIZE):
            batch = missing_queries[i:i + BATCH_SIZE]
            batch_num = (i // BATCH_SIZE) + 1
            
            print(f"\n[{batch_num}/{total_batches}] Downloading Labs batch ({len(batch)} keywords)...")
            
            extracted_data = fetch_dataforseo_labs_volumes(batch, LOCATION_CODE, LANGUAGE_CODE)
            
            if extracted_data:
                df_temp = pd.DataFrame(extracted_data)
                
                df_temp["period"] = pd.to_datetime(df_temp["period"], format="%Y-%m-%d")
                
                df_checkpoint = pd.concat([df_checkpoint, df_temp], ignore_index=True)
                df_checkpoint.to_parquet(CHECKPOINT_FILE)
                print(f"✔ Batch successful. Extracted {len(extracted_data)} rows. Checkpoint updated.")
            else:
                print("! No data returned or parsing failed.")
            
            if batch_num < total_batches:
                time.sleep(5) 
    else:
        print("✔ All keywords are already downloaded.")
else:
    print("RUN_API = False. Change to "True" to execute the extraction.")


[1/1] Downloading Labs batch (220 keywords)...
✔ Batch successful. Extracted 15769 rows. Checkpoint updated.


## 6. Data Quality & Consolidation
Cleans the extracted data and exports the final dataset (search prefix "viaje a " is removed ).

In [ ]:
print("--- DATA QUALITY & CONSOLIDATION ---")

if CHECKPOINT_FILE.exists():
    df_final = pd.read_parquet(CHECKPOINT_FILE)
    print(f"Original Shape: {df_final.shape}")
    
    # 1. Normalize the search term (Remove the query prefix)
    df_final["search_term"] = (
        df_final["search_term"]
        .str.replace("viaje a ", "", regex=False)
        .str.strip()
    )
    
    # 2. Handle duplicated entries
    df_final = df_final.drop_duplicates(subset=["period", "search_term"])
    
    # ---  NEW VALIDATIONS FOR MACHINE LEARNING ---
    print("\n--- QUALITY VALIDATIONS ---")
    
    # A. Total Coverage Validation
    unique_destinations = df_final['search_term'].nunique()
    print(f"Unique destinations downloaded: {unique_destinations} / {len(target_keywords)}")
    
    # B. Null Values Validation
    null_values = df_final.isnull().sum()
    if null_values.sum() > 0:
        print("!! WARNING: Null values (NaNs) detected in the data:")
        print(null_values[null_values > 0])
        # Note: We will fill them with 0 later if needed for the model
    else:
        print("✔ No null values found in the data.")
        
    # C. Temporal Consistency Validation
    months_per_destination = df_final.groupby("search_term").size()
    if months_per_destination.nunique() > 1:
        print("!! WARNING: Temporal asymmetry detected. Not all destinations have the same number of months!")
        print(f"   Minimum months: {months_per_destination.min()} | Maximum months: {months_per_destination.max()}")
    else:
        print(f"✔ Perfect temporal consistency: {months_per_destination.iloc[0]} months for each destination.")


    
    print(f"\nDate Range: {df_final['period'].min().date()} to {df_final['period'].max().date()}")
    
    # 3. Export the clean dataset
    df_final.to_parquet(FINAL_PATH, index=False)
    print(f"\n✔ Final Time-Series Dataset safely exported to: {FINAL_PATH}")
    
    display(df_final.sample(min(10, len(df_final))))
else:
    print("No checkpoint file found. Please run the extraction pipeline first.")

--- DATA QUALITY & CONSOLIDATION ---
Original Shape: (15769, 6)

--- QUALITY VALIDATIONS ---
Unique destinations downloaded: 177 / 220
!! WARNING: Null values (NaNs) detected in the data:
competition          228
competition_level    228
cpc                  319
dtype: int64
!! WARNING: Temporal asymmetry detected. Not all destinations have the same number of months!
   Minimum months: 55 | Maximum months: 99

Date Range: 2018-02-01 to 2026-04-01

✔ Final Time-Series Dataset safely exported to: ..\data\interim\05_dataforseo_monthly__.parquet


,period,search_term,monthly_searches,competition,competition_level,cpc
8174,2025-09-01,jordania,590,0.46,MEDIUM,0.58
5605,2021-08-01,fiji,140,0.38,MEDIUM,0.86
4256,2020-12-01,dubai,1900,0.56,MEDIUM,0.52
15295,2020-10-01,uzbekistan,20,0.49,MEDIUM,0.92
8998,2026-04-01,liechtenstein,90,0.40,MEDIUM,0.69
9745,2022-10-01,maldivas,6600,0.74,HIGH,1.05
4983,2020-06-01,escandinavia,50,0.59,MEDIUM,0.84
10724,2024-10-01,myanmar,320,0.19,LOW,0.73
15332,2025-04-01,venezuela,590,0.57,MEDIUM,0.40
5082,2019-10-01,escocia,2900,0.67,HIGH,0.64


In [26]:
df_final.groupby("search_term")["monthly_searches"].mean().round(0).sort_values(ascending=False).head(50)

search_term
egipto                  17978.0
maldivas                11529.0
marruecos                9853.0
islandia                 8045.0
italia                   7993.0
grecia                   7251.0
bali                     7125.0
portugal                 6198.0
laponia                  6117.0
costa rica               5916.0
tailandia                5004.0
estambul                 4884.0
japon                    4856.0
vietnam                  4794.0
cabo verde               4748.0
turquia                  4662.0
malta                    4591.0
republica dominicana     4471.0
croacia                  4373.0
noruega                  3914.0
mexico                   3733.0
filipinas                3659.0
cuba                     3642.0
dubai                    3510.0
india                    3313.0
escocia                  3124.0
francia                  3040.0
colombia                 3009.0
argentina                2966.0
estados unidos           2906.0
peru                     286

## 7. Appendix: Anchor Keyword Selection for Google Trends
*Note: This section is purely analytical and documents the mathematical selection of the anchor keyword ("viaje a argentina").*

To establish a stable relative index in Google Trends across 200+ destinations, the ideal anchor needed tu fulfill two properties:  
1. **Optimal Volume:** High enough to avoid statistical noise, but not an extreme outlier that would flatten the index for smaller countries (filtered between the 75th and 95th percentiles).
2. **High Temporal Stability:** Minimal seasonal spikes. We calculate the Coefficient of Variation (CV) to find the destination with the flattest, most constant search demand over time.

### 7.1 Final Anchor Keyword Selection

The anchor keyword `"viaje a argentina"` was selected based on:

- **Volume:** ~3,000 monthly searches (75th–95th percentile range)
- **Stability:** CV = 0.279 — the flattest demand curve among candidates

This ensures the anchor neither flattens low-volume destinations nor 
disappears against high-volume ones.

In [ ]:
df_final = pd.read_parquet(FINAL_PATH) 
print("--- KEYWORD ANCHOR SELECTION ---\n")


# Group by keyword and calculate mean and standard deviation
df_stats = df_final.groupby("search_term").agg(
    monthly_mean=("monthly_searches", "mean"),
    monthly_std=("monthly_searches", "std")
).reset_index()


# Calculate the Coefficient of Variation (CV): closer to 0 = higher stability.
df_stats["stability_cv"] = df_stats["monthly_std"] / df_stats["monthly_mean"]

# Volume filter (75th to 95th percentile)
p75 = df_stats["monthly_mean"].quantile(0.75)
p95 = df_stats["monthly_mean"].quantile(0.95)

candidates = df_stats[
    (df_stats["monthly_mean"] >= p75) & 
    (df_stats["monthly_mean"] <= p95)
].copy()

# Sort countries from most to least stable (lowest CV first)
best_anchors = candidates.sort_values("stability_cv", ascending=True)

print(f"Volume filter: Looking for terms between {int(p75)} and {int(p95)} monthly searches.\n")
print(" Top anchors:")
display(best_anchors.round(3).head(30))


--- KEYWORD ANCHOR SELECTION ---

Volume filter: Looking for terms between 1663 and 5956 monthly searches.

 Top anchors:


,search_term,monthly_mean,monthly_std,stability_cv
126,noruega,3914.444,1066.819,0.273
127,nueva zelanda,2035.165,561.422,0.276
7,argentina,2965.934,832.429,0.281
157,suiza,2790.909,845.889,0.303
135,polonia,2099.231,686.602,0.327
141,republica dominicana,4471.429,1469.942,0.329
144,rumania,1787.333,591.030,0.331
115,mexico,3732.967,1236.128,0.331
9,australia,1995.385,665.353,0.333
36,colombia,3009.302,1010.257,0.336
